In [2]:
import os
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    KFold,
    GroupKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
)


# ============================================================
# PROJECT ROOT
# ============================================================

# This notebook is expected to live under:
# <repository>/notebooks/
#
# Therefore, the repository root is the parent directory
# of the notebook directory.

PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is opened with the repository root as the
# Jupyter working directory, PROJECT_ROOT is already correct.
#
# If Jupyter was launched from another directory, walk upward
# until the Git repository is found.

if not (PROJECT_ROOT / ".git").exists():

    current = PROJECT_ROOT

    while current != current.parent:

        if (current / ".git").exists():
            PROJECT_ROOT = current
            break

        current = current.parent


if not (PROJECT_ROOT / ".git").exists():

    raise RuntimeError(
        "Could not locate the project root (.git directory).\n"
        f"Notebook working directory: {Path.cwd()}\n"
        "Please open the notebook from the project repository."
    )


print(f"Project root: {PROJECT_ROOT}")


# ============================================================
# ENVIRONMENT & SEED LOCKING
# ============================================================

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ============================================================
# DIRECTORY HIERARCHY
# ============================================================

BASE_DIR = (
    PROJECT_ROOT
    / "data"
    / "modeling_changes"
    / "baseline_results_v3"
)

BASE_DIR = BASE_DIR.resolve()

SUBDIRS = [
    "metrics",
    "feature_importance",
    "validation",
    "predictions",
    "diagnostics",
    "models",
    "reports",
    "figures",
    "v3_splitting_visualizations",
]

for sub in SUBDIRS:

    (
        BASE_DIR / sub
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# ENVIRONMENT SPECS
# ============================================================

import sklearn
import lightgbm

env_specs = {

    "python_version": sys.version,

    "numpy_version": np.__version__,

    "pandas_version": pd.__version__,

    "sklearn_version": sklearn.__version__,

    "lightgbm_version": lightgbm.__version__,

    "random_seed": RANDOM_SEED,

    "project_root": str(PROJECT_ROOT),

    "working_directory": str(
        Path.cwd().resolve()
    ),
}


with open(
    BASE_DIR
    / "validation"
    / "modeling_environment_v3.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        env_specs,
        f,
        indent=4,
    )


print(
    f"Baseline results directory:\n{BASE_DIR}"
)

print(
    "Environment specification saved successfully."
)

Project root: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11
Baseline results directory:
C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3
Environment specification saved successfully.


In [3]:
# ============================================================
# V3 DATASET LOADING + SPLIT INTEGRITY AUDIT
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "data" / "modeling_changes").exists():
    raise FileNotFoundError(
        f"Could not identify project root.\n"
        f"Detected: {PROJECT_ROOT}\n"
        f"Current directory: {Path.cwd()}"
    )

# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

MASTER_PATH = (
    PROJECT_ROOT
    / "data"
    / "modeling_changes"
    / "datasets"
    / "master_modeling_dataset_v3.csv"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "modeling_changes"
    / "splits"
    / "train.csv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "modeling_changes"
    / "splits"
    / "test.csv"
)

BASE_DIR = (
    PROJECT_ROOT
    / "data"
    / "modeling_changes"
    / "baseline_results_v3"
)

VALIDATION_DIR = BASE_DIR / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. FILE EXISTENCE
# ------------------------------------------------------------

for name, path in {
    "V3 master dataset": MASTER_PATH,
    "V3 train dataset": TRAIN_PATH,
    "V3 test dataset": TEST_PATH,
}.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(f"✓ {name}: {path}")

# ------------------------------------------------------------
# 4. LOAD
# ------------------------------------------------------------

master_df = pd.read_csv(MASTER_PATH)
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("\nLoaded V3 datasets:")
print(f"Master : {master_df.shape}")
print(f"Train  : {train_df.shape}")
print(f"Test   : {test_df.shape}")

# ------------------------------------------------------------
# 5. DERIVE SEASON CONSISTENTLY
# ------------------------------------------------------------

SEASON_MAP = {
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Summer",
    4: "Summer",
    5: "Summer",
    6: "Summer",
    7: "Monsoon",
    8: "Monsoon",
    9: "Monsoon",
    10: "Post-monsoon",
    11: "Post-monsoon",
}

def validate_and_assign_season(df_name, df):
    # Create a de-fragmented copy to prevent Pandas PerformanceWarning
    df = df.copy()
    
    if "season" in df.columns:
        derived_season = df["month"].astype(int).map(SEASON_MAP)
        if not df["season"].equals(derived_season):
            raise AssertionError(
                f"{df_name}: existing 'season' column does not "
                f"match the deterministic month-to-season mapping."
            )
    else:
        df["season"] = df["month"].astype(int).map(SEASON_MAP)

    if df["season"].isna().any():
        raise AssertionError(
            f"{df_name}: season could not be assigned to every row."
        )
        
    return df

# Apply the defragmentation and season assignment safely
master_df = validate_and_assign_season("master", master_df)
train_df = validate_and_assign_season("train", train_df)
test_df = validate_and_assign_season("test", test_df)

# ------------------------------------------------------------
# 6. BASIC COUNTS
# ------------------------------------------------------------

assert len(master_df) == 1615, (
    f"V3 master row mismatch: {len(master_df)}"
)

assert len(train_df) == 1292, (
    f"Train row mismatch: {len(train_df)}"
)

assert len(test_df) == 323, (
    f"Test row mismatch: {len(test_df)}"
)

assert master_df["station"].nunique() == 35
assert train_df["station"].nunique() == 35
assert test_df["station"].nunique() == 34

# ------------------------------------------------------------
# 7. REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "station",
    "year",
    "month",
    "pm25",
]

for name, df in {
    "master": master_df,
    "train": train_df,
    "test": test_df,
}.items():

    missing = [
        c for c in required_columns
        if c not in df.columns
    ]

    assert not missing, (
        f"{name}: missing required columns {missing}"
    )

# ------------------------------------------------------------
# 8. DUPLICATE KEY AUDIT
# ------------------------------------------------------------

KEY_COLUMNS = [
    "station",
    "year",
    "month",
]

for name, df in {
    "master": master_df,
    "train": train_df,
    "test": test_df,
}.items():

    duplicates = df.duplicated(KEY_COLUMNS).sum()

    assert duplicates == 0, (
        f"{name}: {duplicates} duplicate "
        f"station-year-month keys found."
    )

# ------------------------------------------------------------
# 9. KEY UNIVERSE
# ------------------------------------------------------------

def make_keys(df):
    return set(
        zip(
            df["station"].astype(str),
            df["year"].astype(int),
            df["month"].astype(int),
        )
    )

master_keys = make_keys(master_df)
train_keys = make_keys(train_df)
test_keys = make_keys(test_df)

overlap = train_keys & test_keys

assert len(overlap) == 0, (
    f"Train/test overlap detected: {len(overlap)} keys"
)

assert train_keys | test_keys == master_keys, (
    "Train + test key universe does not match V3 master."
)

# ------------------------------------------------------------
# 10. IIT DELHI
# ------------------------------------------------------------

assert "IIT_Delhi" in train_df["station"].values
assert "IIT_Delhi" not in test_df["station"].values

assert (
    (train_df["station"] == "IIT_Delhi").sum() == 1
)

assert (
    (test_df["station"] == "IIT_Delhi").sum() == 0
)

# ------------------------------------------------------------
# 11. YEAR / MONTH VALIDATION
# ------------------------------------------------------------

VALID_YEARS = {2022, 2023, 2024, 2025}

for name, df in {
    "master": master_df,
    "train": train_df,
    "test": test_df,
}.items():

    years = set(df["year"].astype(int).unique())

    assert years.issubset(VALID_YEARS), (
        f"{name}: invalid year(s): {years - VALID_YEARS}"
    )

    assert df["month"].astype(int).between(1, 12).all()

# ------------------------------------------------------------
# 12. PM2.5 + NUMERIC INTEGRITY
# ------------------------------------------------------------

for name, df in {
    "master": master_df,
    "train": train_df,
    "test": test_df,
}.items():

    assert not df["pm25"].isna().any(), (
        f"{name}: missing PM2.5 values."
    )

    numeric = df.select_dtypes(include=np.number)

    assert not np.isinf(numeric).any().any(), (
        f"{name}: infinite numeric values detected."
    )

# ------------------------------------------------------------
# 13. SCHEMA CHECK
# ------------------------------------------------------------
# season is a derived analytical column, so it should now be
# present consistently in all three datasets.

master_schema = list(master_df.columns)
train_schema = list(train_df.columns)
test_schema = list(test_df.columns)

assert master_schema == train_schema, (
    "Master and train schemas still differ."
)

assert master_schema == test_schema, (
    "Master and test schemas still differ."
)

# ------------------------------------------------------------
# 14. TRAIN/TEST DISTRIBUTION AUDIT
# ------------------------------------------------------------

numeric_cols = [
    c
    for c in train_df.select_dtypes(include=np.number).columns
    if c != "pm25"
]

dist_audit = []

for col in numeric_cols:

    tr_mean = train_df[col].mean()
    tr_std = train_df[col].std()

    te_mean = test_df[col].mean()
    te_std = test_df[col].std()

    pooled_std = np.sqrt(
        (
            np.nanvar(train_df[col], ddof=1)
            +
            np.nanvar(test_df[col], ddof=1)
        ) / 2
    )

    smd = (
        abs(tr_mean - te_mean) / pooled_std
        if pooled_std > 0
        else 0.0
    )

    dist_audit.append({
        "feature": col,
        "train_mean": tr_mean,
        "train_std": tr_std,
        "test_mean": te_mean,
        "test_std": te_std,
        "absolute_mean_difference": abs(
            tr_mean - te_mean
        ),
        "standardized_mean_difference": smd,
    })

dist_audit_df = pd.DataFrame(dist_audit)

dist_audit_df.to_csv(
    VALIDATION_DIR
    / "train_test_feature_distribution_audit.csv",
    index=False,
)

# ------------------------------------------------------------
# 15. SAVE AUDIT
# ------------------------------------------------------------

audit_summary = {
    "dataset": "V3",
    "master_rows": len(master_df),
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "master_stations": int(master_df["station"].nunique()),
    "train_stations": int(train_df["station"].nunique()),
    "test_stations": int(test_df["station"].nunique()),
    "train_test_key_overlap": len(overlap),
    "union_matches_master": True,
    "iit_delhi_train": 1,
    "iit_delhi_test": 0,
    "duplicate_keys_master": 0,
    "duplicate_keys_train": 0,
    "duplicate_keys_test": 0,
    "validation_status": "PASS",
}

with open(
    VALIDATION_DIR / "v3_data_integrity_audit.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(audit_summary, f, indent=4)

# ------------------------------------------------------------
# 16. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V3 DATA LOADING + INTEGRITY AUDIT: PASS")
print("=" * 70)
print(f"Project root : {PROJECT_ROOT}")
print(f"Master       : {master_df.shape}")
print(f"Train        : {train_df.shape}")
print(f"Test         : {test_df.shape}")
print(f"Schema       : {len(master_df.columns)} columns")
print(f"Key overlap  : {len(overlap)}")
print("IIT_Delhi    : 1 train / 0 test")
print(f"Audit        : {VALIDATION_DIR}")
print("=" * 70)

✓ V3 master dataset: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\datasets\master_modeling_dataset_v3.csv
✓ V3 train dataset: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\splits\train.csv
✓ V3 test dataset: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\splits\test.csv

Loaded V3 datasets:
Master : (1615, 196)
Train  : (1292, 197)
Test   : (323, 197)

V3 DATA LOADING + INTEGRITY AUDIT: PASS
Project root : C:\Users\Hitakkshi Joshi\Desktop\acm slot 11
Master       : (1615, 197)
Train        : (1292, 197)
Test         : (323, 197)
Schema       : 197 columns
Key overlap  : 0
IIT_Delhi    : 1 train / 0 test
Audit        : C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\validation


In [4]:
TARGET = "pm25"
METADATA_EXCLUDES = [
    "station", "date", "year_month", "season", TARGET,
    "pm25_predicted", "pm25_residuals", "pm25_lag1"
]

# Identify Valid Predictors dynamically
predictors = [col for col in train_df.columns if col not in METADATA_EXCLUDES and not col.startswith("target_")]

# Confirm exact feature types and exclusions
explicit_included_diagnostics = {
    "year_included": "year" in predictors,
    "month_included": "month" in predictors,
    "latitude_included": "latitude" in predictors or "lat" in predictors,
    "longitude_included": "longitude" in predictors or "lon" in predictors,
    "season_as_predictor": "season" in predictors
}

with open(f"{BASE_DIR}/validation/feature_leakage_audit.json", "w") as f:
    json.dump({"predictors_used": predictors, "diagnostics": explicit_included_diagnostics}, f, indent=4)

# Dynamic Feature Group Classification
def classify_feature(col_name):
    c = col_name.lower()
    if any(k in c for k in ["ndvi", "evi", "ndwi", "green", "veg", "nir"]):
        return "Green_Cover"
    elif any(k in c for k in ["temp", "lst", "precip", "wind", "humidity", "era5", "pressure"]):
        return "Meteorology"
    elif any(k in c for k in ["no2", "so2", "aod", "s5p", "pollution"]):
        return "Pollution_Anthropogenic"
    elif any(k in c for k in ["pop", "density", "worldpop"]):
        return "Population"
    elif any(k in c for k in ["road", "osm", "highway", "dist_primary"]):
        return "Road_Infrastructure"
    elif any(k in c for k in ["dw_", "landcover", "dynamic_world", "built", "water"]):
        return "Land_Cover"
    elif any(k in c for k in ["year", "month", "lat", "lon", "sin", "cos", "x", "y"]):
        return "Spatial_Temporal"
    else:
        return "Other"

# DEFINE feature_groups FIRST
feature_groups = {col: classify_feature(col) for col in predictors}

# THEN filter for unclassified features
unclassified = [
    c for c, g in feature_groups.items()
    if g == "Other"
]

print("Unclassified features:")
print(unclassified)

X_train, y_train = train_df[predictors].copy(), train_df[TARGET].values
X_test, y_test = test_df[predictors].copy(), test_df[TARGET].values

Unclassified features:
['season_encoded']


In [5]:
# Models Definition
models = {
    "Linear_Regression": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    "Ridge_Regression": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(random_state=RANDOM_SEED))
    ]),
    "Random_Forest": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1))
    ]),
    "LightGBM": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', LGBMRegressor(random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1))
    ])
}

# Search Grids (Training Set Only)
param_grids = {
    "Linear_Regression": {},
    "Ridge_Regression": {'model__alpha': [0.1, 1.0, 10.0, 100.0, 500.0]},
    "Random_Forest": {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [10, 20, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', 1.0]
    },
    "LightGBM": {
        'model__n_estimators': [100, 200, 300],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__num_leaves': [15, 31, 63],
        'model__max_depth': [-1, 10, 20],
        'model__subsample': [0.7, 0.8, 1.0],
        'model__colsample_bytree': [0.7, 0.8, 1.0],
        'model__reg_lambda': [0.0, 1.0, 5.0]
    }
}

best_estimators = {}
best_params_summary = {}
from sklearn.model_selection import RandomizedSearchCV

best_estimators = {}
best_params_summary = {}

import os
from sklearn.model_selection import RandomizedSearchCV, ParameterGrid

best_estimators = {}
best_params_summary = {}

# Ensure target directory exists prior to saving
metrics_dir = f"{BASE_DIR}/metrics"
os.makedirs(metrics_dir, exist_ok=True)

# 5-Fold Standard CV Tuning
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

for name, pipe in models.items():
    print(f"Tuning {name}...")
    grid = param_grids[name]
    
    if grid:
        # Prevent UserWarning by capping n_iter to max possible combinations
        total_combinations = len(ParameterGrid(grid))
        n_iter = min(15, total_combinations)

        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=grid,
            n_iter=n_iter,
            scoring='neg_root_mean_squared_error',
            cv=kf,
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
        search.fit(X_train, y_train)
        best_estimators[name] = search.best_estimator_
        best_params_summary[name] = search.best_params_
    else:
        pipe.fit(X_train, y_train)
        best_estimators[name] = pipe
        best_params_summary[name] = {"info": "default_ols_no_tuning"}

# Convert summary to DataFrame
df_best_params = pd.DataFrame.from_dict(best_params_summary, orient='index')

# Save output safely to disk
df_best_params.to_csv(os.path.join(metrics_dir, "best_hyperparameters.csv"))

best_params_summary_clean = {
    name: (
        params
        if "info" not in params
        else params
    )
    for name, params in best_params_summary.items()
}

for name, params in best_params_summary_clean.items():
    print(f"\n{name}")
    for key, value in params.items():
        print(f"  {key}: {value}")

Tuning Linear_Regression...
Tuning Ridge_Regression...
Tuning Random_Forest...
Tuning LightGBM...

Linear_Regression
  info: default_ols_no_tuning

Ridge_Regression
  model__alpha: 500.0

Random_Forest
  model__n_estimators: 300
  model__min_samples_split: 2
  model__min_samples_leaf: 2
  model__max_features: 1.0
  model__max_depth: None

LightGBM
  model__subsample: 0.8
  model__reg_lambda: 5.0
  model__num_leaves: 15
  model__n_estimators: 300
  model__max_depth: 20
  model__learning_rate: 0.05
  model__colsample_bytree: 0.7


In [6]:
grouped_cv_results = []

for name, model in best_estimators.items():
    # A. Shuffled 5-Fold CV
    cv_r2, cv_rmse, cv_mae = [], [], []
    for train_idx, val_idx in kf.split(X_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        cv_r2.append(r2_score(y_va, preds))
        cv_rmse.append(np.sqrt(mean_squared_error(y_va, preds)))
        cv_mae.append(mean_absolute_error(y_va, preds))
        
    # B. Spatial Grouped CV (Station)
    sgkf = GroupKFold(n_splits=5)
    spatial_r2 = []
    for train_idx, val_idx in sgkf.split(X_train, y_train, train_df['station']):
        X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        spatial_r2.append(r2_score(y_va, preds))
        
    # C. Temporal Grouped CV (Year)
    temporal_r2 = []
    years = sorted(train_df["year"].unique())
    for yr in years:
        tr_mask = train_df['year'] != yr
        va_mask = train_df['year'] == yr
        model.fit(X_train[tr_mask], y_train[tr_mask])
        preds = model.predict(X_train[va_mask])
        temporal_r2.append(r2_score(y_train[va_mask], preds))

    grouped_cv_results.append({
        "Model": name,
        "CV_R2_Mean": np.mean(cv_r2), "CV_R2_STD": np.std(cv_r2),
        "CV_RMSE_Mean": np.mean(cv_rmse), "CV_RMSE_STD": np.std(cv_rmse),
        "CV_MAE_Mean": np.mean(cv_mae), "CV_MAE_STD": np.std(cv_mae),
        "Spatial_CV_Mean_R2": np.mean(spatial_r2),
        "Temporal_CV_Mean_R2": np.mean(temporal_r2)
    })

pd.DataFrame(grouped_cv_results).to_csv(f"{BASE_DIR}/metrics/grouped_cv_performance_v3.csv", index=False)

In [7]:
predictions_dict = {
    "station": test_df['station'].values,
    "year": test_df['year'].values,
    "month": test_df['month'].values,
    "pm25_actual": y_test
}

model_metrics = []

for name, model in best_estimators.items():
    # Fit full training set, predict holdout test set
    model.fit(X_train, y_train)
    tr_preds = model.predict(X_train)
    te_preds = model.predict(X_test)
    
    predictions_dict[f"pred_{name}"] = te_preds
    
    tr_r2 = r2_score(y_train, tr_preds)
    te_r2 = r2_score(y_test, te_preds)
    rmse = np.sqrt(mean_squared_error(y_test, te_preds))
    mae = mean_absolute_error(y_test, te_preds)
    medae = median_absolute_error(y_test, te_preds)
    
    g_cv = [g for g in grouped_cv_results if g["Model"] == name][0]
    
    model_metrics.append({
        "Model": name,
        "Train_R2": tr_r2,
        "Test_R2": te_r2,
        "Test_RMSE": rmse,
        "Test_MAE": mae,
        "Test_MedianAE": medae,
        "CV_R2_Mean": g_cv["CV_R2_Mean"],
        "CV_R2_STD": g_cv["CV_R2_STD"],
        "CV_RMSE_Mean": g_cv["CV_RMSE_Mean"],
        "CV_RMSE_STD": g_cv["CV_RMSE_STD"],
        "CV_MAE_Mean": g_cv["CV_MAE_Mean"],
        "CV_MAE_STD": g_cv["CV_MAE_STD"],
        "R2_Gap": tr_r2 - te_r2
    })

comparison_df = pd.DataFrame(model_metrics)
comparison_df.to_csv(f"{BASE_DIR}/metrics/model_comparison_v3.csv", index=False)

# Automatic Selection Rule: Best Test R2 with cross-validation check
best_model_name = comparison_df.sort_values(by="Test_R2", ascending=False).iloc[0]["Model"]
best_model_pipe = best_estimators[best_model_name]

# Save Predictions & Residuals
pred_df = pd.DataFrame(predictions_dict)
pred_df['best_model_residual'] = pred_df['pm25_actual'] - pred_df[f"pred_{best_model_name}"]
pred_df.to_csv(f"{BASE_DIR}/predictions/test_predictions_v3.csv", index=False)

In [8]:
# 1. Year-Wise Performance
year_perf = []
for yr in sorted(test_df['year'].unique()):
    idx = test_df['year'] == yr
    actuals = y_test[idx]
    for name in models.keys():
        preds = pred_df.loc[idx, f"pred_{name}"]
        res = actuals - preds
        year_perf.append({
            "Model": name, "Year": yr, "N": idx.sum(),
            "Mean_Actual": np.mean(actuals), "Mean_Predicted": np.mean(preds),
            "Mean_Residual": np.mean(res), "R2": r2_score(actuals, preds),
            "RMSE": np.sqrt(mean_squared_error(actuals, preds)),
            "MAE": mean_absolute_error(actuals, preds),
            "MedianAE": median_absolute_error(actuals, preds)
        })
pd.DataFrame(year_perf).to_csv(f"{BASE_DIR}/metrics/yearwise_performance_v3.csv", index=False)

# 2. Season-Wise Performance
season_perf = []
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5, 6]:
        return "Summer"
    elif month in [7, 8, 9]:
        return "Monsoon"
    elif month in [10, 11]:
        return "Post-monsoon"
    return "Unknown"

test_seasons = test_df["month"].map(get_season)

for ssn in ['Winter', 'Summer', 'Monsoon', 'Post-monsoon']:
    idx = test_seasons == ssn
    if idx.sum() > 0:
        actuals = y_test[idx]
        for name in models.keys():
            preds = pred_df.loc[idx, f"pred_{name}"]
            season_perf.append({
                "Model": name, "Season": ssn, "N": idx.sum(),
                "R2": r2_score(actuals, preds),
                "RMSE": np.sqrt(mean_squared_error(actuals, preds)),
                "MAE": mean_absolute_error(actuals, preds),
                "MedianAE": median_absolute_error(actuals, preds)
            })
pd.DataFrame(season_perf).to_csv(f"{BASE_DIR}/metrics/seasonwise_performance_v3.csv", index=False)

# 3. Residual Summary Diagnostics for Best Model
best_res = pred_df['best_model_residual']
res_summary = {
    "best_model": best_model_name,
    "mean_residual": np.mean(best_res),
    "median_residual": np.median(best_res),
    "std_residual": np.std(best_res),
    "skewness": float(pd.Series(best_res).skew()),
    "max_underprediction": float(np.max(best_res)), # actual >> pred
    "max_overprediction": float(np.min(best_res))   # actual << pred
}
with open(f"{BASE_DIR}/diagnostics/residual_summary_v3.json", "w") as f:
    json.dump(res_summary, f, indent=4)

In [9]:
# Compute and Save Model-Specific Feature Importances
group_imp_records = []

for name, pipe in best_estimators.items():
    model_obj = pipe.named_steps['model']
    if hasattr(model_obj, 'feature_importances_'):
        imps = model_obj.feature_importances_
    elif hasattr(model_obj, 'coef_'):
        imps = np.abs(model_obj.coef_)
    else:
        continue
        
    imps_norm = imps / np.sum(imps)
    feat_df = pd.DataFrame({"feature": predictors, "importance": imps_norm})
    feat_df['group'] = feat_df['feature'].map(feature_groups)
    feat_df.to_csv(f"{BASE_DIR}/feature_importance/importance_{name}_v3.csv", index=False)
    
    # Aggregate by conceptual feature group
    grp = feat_df.groupby('group')['importance'].sum().reset_index()
    grp['Model'] = name
    group_imp_records.append(grp)

pd.concat(group_imp_records).to_csv(f"{BASE_DIR}/feature_importance/feature_group_importance_v3.csv", index=False)

In [10]:
# ============================================================
# V3 MODEL VISUALIZATION CELL
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Plot configuration
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

FIG_DIR = Path(
    BASE_DIR
) / "v3_splitting_visualizations"

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Basic synchronization checks
# ------------------------------------------------------------

required_objects = [
    "comparison_df",
    "pred_df",
    "best_model_name",
    "test_df",
    "train_df",
    "predictors",
    "feature_groups",
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The visualization cell is missing required objects: "
        + ", ".join(missing_objects)
    )


BEST_PRED_COL = (
    f"pred_{best_model_name}"
)

if BEST_PRED_COL not in pred_df.columns:
    raise RuntimeError(
        f"Prediction column '{BEST_PRED_COL}' "
        "was not found in pred_df."
    )

if "pm25_actual" not in pred_df.columns:
    raise RuntimeError(
        "'pm25_actual' is missing from pred_df."
    )


# ------------------------------------------------------------
# Helper: safe figure save
# ------------------------------------------------------------

def save_figure(fig, filename):
    path = FIG_DIR / filename

    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    print(f"Saved: {path}")


# ============================================================
# FIGURE 1 — MODEL PERFORMANCE
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 5.5),
    dpi=300,
)

df_fig1 = comparison_df[
    [
        "Model",
        "Test_R2",
        "Test_RMSE",
        "Test_MAE",
    ]
].melt(
    id_vars=["Model"],
    value_vars=[
        "Test_R2",
        "Test_RMSE",
        "Test_MAE",
    ],
    var_name="Metric",
    value_name="Value",
)

metric_labels = {
    "Test_R2": "R²",
    "Test_RMSE": "RMSE",
    "Test_MAE": "MAE",
}

df_fig1["Metric"] = (
    df_fig1["Metric"]
    .map(metric_labels)
)

sns.barplot(
    data=df_fig1,
    x="Metric",
    y="Value",
    hue="Model",
    palette="viridis",
    ax=ax,
)

ax.set_title(
    "Predictive Baseline Performance Comparison",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel("Evaluation Metric")

ax.set_ylabel(
    "Metric Value"
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    title="Model",
    frameon=True,
)

save_figure(
    fig,
    "fig1_model_performance_comparison_v3.png",
)


# ============================================================
# FIGURE 2 — OBSERVED VS PREDICTED
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.5, 6.5),
    dpi=300,
)

actual = (
    pred_df["pm25_actual"]
    .astype(float)
)

predicted = (
    pred_df[BEST_PRED_COL]
    .astype(float)
)

plot_min = min(
    actual.min(),
    predicted.min(),
)

plot_max = max(
    actual.max(),
    predicted.max(),
)

sns.scatterplot(
    x=actual,
    y=predicted,
    alpha=0.70,
    s=45,
    color="#2b5c8f",
    edgecolor="white",
    linewidth=0.4,
    label="Test observations",
    ax=ax,
)

ax.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    linestyle="--",
    linewidth=2,
    color="black",
    label="1:1 reference",
)

# Trend line without adding an unnecessary legend label.
sns.regplot(
    x=actual,
    y=predicted,
    scatter=False,
    color="#d95f02",
    line_kws={
        "linewidth": 2,
        "linestyle": ":",
    },
    ax=ax,
)

ax.set_title(
    f"Observed vs Predicted PM₂.₅ — {best_model_name}",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Observed PM₂.₅ (μg/m³)"
)

ax.set_ylabel(
    "Predicted PM₂.₅ (μg/m³)"
)

ax.grid(
    linestyle="--",
    alpha=0.3,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    loc="upper left",
    frameon=True,
)

save_figure(
    fig,
    "fig2_observed_predicted_range_v3.png",
)


# ============================================================
# FIGURE 3 — SEASON × YEAR RMSE
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5.5),
    dpi=300,
)

heatmap_df = pred_df[
    [
        "pm25_actual",
        BEST_PRED_COL,
    ]
].copy()

# Align year/season explicitly by position.
heatmap_df["year"] = (
    test_df["year"]
    .reset_index(drop=True)
)

if "season" in test_df.columns:

    heatmap_df["season"] = (
        test_df["season"]
        .reset_index(drop=True)
    )

else:

    heatmap_df["season"] = (
        test_df["month"]
        .map(
            {
                1: "Winter",
                2: "Winter",
                3: "Summer",
                4: "Summer",
                5: "Summer",
                6: "Summer",
                7: "Monsoon",
                8: "Monsoon",
                9: "Monsoon",
                10: "Post-monsoon",
                11: "Post-monsoon",
                12: "Winter",
            }
        )
        .reset_index(drop=True)
    )
season_rmse = (
    heatmap_df
    .groupby(
        ["season", "year"]
    )
    .apply(
        lambda g: np.sqrt(
            mean_squared_error(
                g["pm25_actual"],
                g[BEST_PRED_COL],
            )
        ),
        include_groups=False,
    )
    .unstack()
)

SEASON_ORDER = [
    "Winter",
    "Summer",
    "Monsoon",
    "Post-monsoon",
]

season_rmse = season_rmse.reindex(
    SEASON_ORDER
)

sns.heatmap(
    season_rmse,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "RMSE (μg/m³)"
    },
    ax=ax,
)

ax.set_title(
    f"Prediction Error by Season and Year — {best_model_name}",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Study Year"
)

ax.set_ylabel(
    "Season"
)

save_figure(
    fig,
    "fig3_year_season_rmse_heatmap_v3.png",
)


# ============================================================
# FIGURE 4 — SPATIAL STATION ERROR
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5),
    dpi=300,
)

error_df = pred_df[
    [
        "station",
        "pm25_actual",
        BEST_PRED_COL,
    ]
].copy()

error_df["squared_error"] = (
    error_df["pm25_actual"]
    - error_df[BEST_PRED_COL]
) ** 2

station_error = (
    error_df
    .groupby("station")
    .agg(
        rmse=(
            "squared_error",
            lambda x: np.sqrt(
                x.mean()
            ),
        ),
        count=(
            "pm25_actual",
            "size",
        ),
    )
    .reset_index()
)

if (
    "latitude" in test_df.columns
    and
    "longitude" in test_df.columns
):

    coordinates = (
        test_df[
            [
                "station",
                "latitude",
                "longitude",
            ]
        ]
        .drop_duplicates(
            subset=["station"]
        )
    )

    station_error = (
        station_error.merge(
            coordinates,
            on="station",
            how="left",
        )
    )

    scatter = ax.scatter(
        station_error["longitude"],
        station_error["latitude"],
        c=station_error["rmse"],
        s=np.maximum(
            station_error["count"] * 22,
            50,
        ),
        cmap="magma",
        alpha=0.88,
        edgecolors="black",
        linewidths=0.6,
    )

    colorbar = fig.colorbar(
        scatter,
        ax=ax,
    )

    colorbar.set_label(
        "Station RMSE (μg/m³)"
    )

    top_error = (
        station_error
        .sort_values(
            "rmse",
            ascending=False,
        )
        .head(3)
    )

    for _, row in top_error.iterrows():

        ax.annotate(
            row["station"],
            (
                row["longitude"],
                row["latitude"],
            ),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=8,
            fontweight="bold",
        )

else:

    ax.text(
        0.5,
        0.5,
        "Coordinates unavailable for spatial visualization.",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

ax.set_title(
    f"Spatial Distribution of Prediction Error — {best_model_name}",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Longitude"
)

ax.set_ylabel(
    "Latitude"
)

ax.grid(
    linestyle="--",
    alpha=0.25,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_figure(
    fig,
    "fig4_spatial_station_error_v3.png",
)


# ============================================================
# FIGURE 5 — GREEN COVER VS PM₂.₅
# ============================================================
%pip install statsmodels

green_cols = [
    col
    for col in predictors
    if feature_groups.get(
        col
    ) == "Green_Cover"
]

# More specific candidate list.
green_candidates = [
    col
    for col in green_cols
    if any(
        term in col.lower()
        for term in [
            "ndvi",
            "evi",
            "green",
            "trees",
            "vegetation",
        ]
    )
]

# ------------------------------------------------------------
# Automatically select ONE representative green-cover proxy.
# Priority: Sentinel-2 NDVI mean 1000m → 500m → EVI.
# ------------------------------------------------------------

priority_patterns = [
    lambda c: (
        c.startswith("sentinel2_ndvi_mean_1000m")
    ),
    lambda c: (
        c.startswith("sentinel2_ndvi_mean_500m")
    ),
    lambda c: (
        c.startswith("sentinel2_evi_mean_1000m")
    ),
    lambda c: (
        c.startswith("sentinel2_evi_mean_500m")
    ),
    lambda c: (
        "ndvi_mean_1000m" in c.lower()
    ),
    lambda c: (
        "ndvi_mean_500m" in c.lower()
    ),
    lambda c: (
        "evi_mean_1000m" in c.lower()
    ),
    lambda c: (
        "evi_mean_500m" in c.lower()
    ),
]

target_green = None

for pattern in priority_patterns:

    matches = [
        col
        for col in green_candidates
        if pattern(col)
    ]

    if matches:

        target_green = sorted(
            matches
        )[0]

        break

if target_green is None and green_candidates:

    target_green = sorted(
        green_candidates
    )[0]


fig, ax = plt.subplots(
    figsize=(8.5, 5.5),
    dpi=300,
)
if target_green is not None:

    x_green = (
        test_df[target_green]
        .reset_index(drop=True)
        .astype(float)
    )

    y_actual = (
        pred_df["pm25_actual"]
        .reset_index(drop=True)
        .astype(float)
    )

    y_pred = (
        pred_df[BEST_PRED_COL]
        .reset_index(drop=True)
        .astype(float)
    )

    # Keep only complete observations.
    valid_mask = (
        x_green.notna()
        & y_actual.notna()
        & y_pred.notna()
    )

    x_green = x_green[valid_mask]
    y_actual = y_actual[valid_mask]
    y_pred = y_pred[valid_mask]

    # Observed PM2.5 scatter.
    sns.scatterplot(
        x=x_green,
        y=y_actual,
        alpha=0.58,
        s=42,
        color="#777777",
        edgecolor="white",
        linewidth=0.3,
        label="Observed PM₂.₅",
        ax=ax,
    )

    # Smooth predictive relationship.
    if len(x_green) >= 5:

        sns.regplot(
            x=x_green,
            y=y_pred,
            scatter=False,
            lowess=True,
            color="#2ca25f",
            line_kws={
                "linewidth": 2.5,
            },
            ax=ax,
        )

        # Add a clean legend entry for the smooth curve.
        ax.plot(
            [],
            [],
            color="#2ca25f",
            linewidth=2.5,
            label="LOESS fit of model predictions",
        )

    ax.set_title(
        "Green Cover Proxy and PM₂.₅",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        target_green
    )

    ax.set_ylabel(
        "PM₂.₅ Concentration (μg/m³)"
    )

    ax.grid(
        linestyle="--",
        alpha=0.3,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        frameon=True,
    )

else:

    ax.text(
        0.5,
        0.5,
        "No suitable green-cover predictor was found.",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

    ax.set_title(
        "Green Cover Proxy and PM₂.₅",
        fontsize=13,
        fontweight="bold",
    )

save_figure(
    fig,
    "fig5_green_cover_relationship_v3.png",
)


# ============================================================
# FIGURE 6 — FEATURE GROUP IMPORTANCE
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 5.5),
    dpi=300,
)

importance_path = (
    Path(BASE_DIR)
    / "feature_importance"
    / "feature_group_importance_v3.csv"
)

if importance_path.exists():

    group_importance = pd.read_csv(
        importance_path
    )

    group_importance = (
        group_importance[
            group_importance["Model"]
            == best_model_name
        ]
        .sort_values(
            "importance",
            ascending=True,
        )
    )

    if not group_importance.empty:

        ax.barh(
            group_importance["group"],
            group_importance["importance"],
            color="#377eb8",
            alpha=0.9,
        )

        ax.set_title(
            f"Aggregated Feature-Group Importance — {best_model_name}",
            fontsize=13,
            fontweight="bold",
            pad=12,
        )

        ax.set_xlabel(
            "Aggregated Predictive Importance"
        )

        ax.set_ylabel(
            "Feature Group"
        )

        ax.grid(
            axis="x",
            linestyle="--",
            alpha=0.3,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    else:

        ax.text(
            0.5,
            0.5,
            f"No feature-group results found for {best_model_name}.",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

else:

    ax.text(
        0.5,
        0.5,
        "Feature-group importance file not found.",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

save_figure(
    fig,
    "fig6_feature_group_importance_v3.png",
)


# ============================================================
# FIGURE 7 — TRAIN / TEST YEAR AND MONTH DISTRIBUTION
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
    dpi=300,
)

sns.countplot(
    data=train_df,
    x="year",
    color="#4c78a8",
    ax=axes[0],
)

sns.countplot(
    data=test_df,
    x="year",
    color="#f58518",
    ax=axes[0],
)

axes[0].set_title(
    "Train/Test Distribution by Year",
    fontsize=11,
    fontweight="bold",
)

axes[0].set_xlabel(
    "Study Year"
)

axes[0].set_ylabel(
    "Number of observations"
)

axes[0].legend(
    ["Train", "Test"]
)

axes[0].grid(
    axis="y",
    linestyle="--",
    alpha=0.3,
)


sns.countplot(
    data=train_df,
    x="month",
    color="#4c78a8",
    ax=axes[1],
)

sns.countplot(
    data=test_df,
    x="month",
    color="#f58518",
    ax=axes[1],
)

axes[1].set_title(
    "Train/Test Distribution by Month",
    fontsize=11,
    fontweight="bold",
)

axes[1].set_xlabel(
    "Month"
)

axes[1].set_ylabel(
    "Number of observations"
)

axes[1].legend(
    ["Train", "Test"]
)

axes[1].grid(
    axis="y",
    linestyle="--",
    alpha=0.3,
)

for axis in axes:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

plt.tight_layout()

save_figure(
    fig,
    "v3_split_distribution_summary.png",
)


print(
    "\nV3 visualization generation completed successfully."
)

print(
    f"Output directory: {FIG_DIR}"
)

Saved: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\v3_splitting_visualizations\fig1_model_performance_comparison_v3.png
Saved: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\v3_splitting_visualizations\fig2_observed_predicted_range_v3.png
Saved: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\v3_splitting_visualizations\fig3_year_season_rmse_heatmap_v3.png
Saved: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\v3_splitting_visualizations\fig4_spatial_station_error_v3.png
Note: you may need to restart the kernel to use updated packages.
Saved: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\v3_splitting_visualizations\fig5_green_cover_relationship_v3.png
Saved: C:\Users\Hitakkshi Joshi\Desktop\acm slot 11\data\modeling_changes\baseline_results_v3\v3_splitting_visualizations\fig6_feature_gro